In [1]:
import os
os.chdir('/home/smallyan/relation_eval_agent')

# Inherit environment from bashrc
import subprocess
result = subprocess.run(['bash', '-c', 'source /home/smallyan/.bashrc && env'], capture_output=True, text=True)
for line in result.stdout.split('\n'):
    if '=' in line:
        key, _, value = line.partition('=')
        os.environ[key] = value

# Set HF_HOME
os.environ['HF_HOME'] = '/net/projects2/chai-lab/shared_models'

print("Working directory:", os.getcwd())
print("HF_HOME:", os.environ.get('HF_HOME'))
print("CUDA available:", end=" ")

import torch
print(torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))

Working directory: /home/smallyan/relation_eval_agent
HF_HOME: /net/projects2/chai-lab/shared_models
CUDA available: 

True
CUDA device: NVIDIA A100 80GB PCIe


# Generalizability Evaluation for InterpDetect Repository

## Overview
This notebook evaluates the generalizability of findings in the InterpDetect repository using the Generalizability Checklist:
- **GT1**: Model Generalization
- **GT2**: Data Generalization  
- **GT3**: Method/Specificity Generalizability

## Repository Location
`/net/scratch2/smallyan/InterpDetect_eval`

In [2]:
# Explore the repository structure
repo_path = '/net/scratch2/smallyan/InterpDetect_eval'

import os
for root, dirs, files in os.walk(repo_path):
    # Limit depth
    level = root.replace(repo_path, '').count(os.sep)
    if level < 3:
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/')
        subindent = ' ' * 2 * (level + 1)
        for file in files:
            print(f'{subindent}{file}')

InterpDetect_eval/
  documentation.pdf
  plan.md
  .gitignore
  CodeWalkthrough.md
  LICENSE
  requirements.txt
  trained_models/
    model_RandomForest_3000.pickle
    model_LR_3000.pickle
    model_SVC_3000.pickle
    model_XGBoost_3000.pickle
  .git/
    config
    packed-refs
    index
    description
    HEAD
    FETCH_HEAD
    ORIG_HEAD
    COMMIT_EDITMSG
    logs/
      HEAD
    refs/
    objects/
    hooks/
      pre-rebase.sample
      sendemail-validate.sample
      post-update.sample
      pre-receive.sample
      commit-msg.sample
      pre-push.sample
      fsmonitor-watchman.sample
      push-to-checkout.sample
      prepare-commit-msg.sample
      pre-merge-commit.sample
      pre-commit.sample
      pre-applypatch.sample
      applypatch-msg.sample
      update.sample
    info/
      exclude
  evaluation/
    self_matching.ipynb
    code_critic_evaluation.ipynb
    consistency_evaluation.json
    code_critic_summary.json
    replication_eval/
      documentation_eval_su

In [3]:
# Read the plan.md to understand the research
with open(os.path.join(repo_path, 'plan.md'), 'r') as f:
    plan_content = f.read()
print(plan_content)

# Plan
## Objective
Develop a mechanistic interpretability-based hallucination detection method for Retrieval-Augmented Generation (RAG) systems by computing External Context Scores (ECS) across layers and attention heads and Parametric Knowledge Scores (PKS) across layers (FFN), training regression-based classifiers on these signals, and demonstrating generalization from a small proxy model (Qwen3-0.6b) to larger production models (GPT-4.1-mini).

## Hypothesis
1. RAG hallucinations correlate with:  later-layer FFN modules disproportionately inject parametric knowledge into the residual stream while attention heads fail to adequately exploit external context.
2. External Context Score (ECS) and Parametric Knowledge Score (PKS) are correlated with hallucination occurrence and can serve as predictive features for hallucination detection.
3. Mechanistic signals extracted from a small proxy model (0.6b parameters) can generalize to detect hallucinations in responses from larger production

In [4]:
# Read CodeWalkthrough.md to understand the implementation
with open(os.path.join(repo_path, 'CodeWalkthrough.md'), 'r') as f:
    walkthrough_content = f.read()
print(walkthrough_content)

# InterpDetect

**InterpDetect: Interpretable Signals for Detecting Hallucinations in Retrieval-Augmented Generation**

[![Paper](https://img.shields.io/badge/Paper-OpenReview-blue)](https://openreview.net/pdf?id=TZzBKwHLwF)
[![License](https://img.shields.io/badge/License-MIT-green.svg)](LICENSE)

A comprehensive framework for detecting and analyzing hallucinations in Retrieval-Augmented Generation (RAG) systems using interpretability techniques and chunk-level analysis on the RAGBench/FinQA dataset.

## Overview

This project implements a novel approach to hallucination detection by leveraging interpretability methods. The framework consists of three main parts:

1. **Preprocessing Pipeline** - Convert raw datasets to labeled data with hallucination spans
2. **Training & Prediction** - Train classifiers or use pre-trained models for hallucination detection
3. **Baseline Comparisons** - Evaluate against multiple baseline methods (RAGAS, TruLens, RefChecker, GPT-based, etc.)

The frame

In [5]:
# Read the compute_scores.py to understand ECS and PKS computation
with open(os.path.join(repo_path, 'scripts/compute_scores.py'), 'r') as f:
    compute_scores_content = f.read()
print(compute_scores_content)

# %%
#!pip install transformer_lens

import torch
from transformers import AutoTokenizer
from transformer_lens import HookedTransformer
import json
from torch.nn import functional as F
from typing import Dict, List, Tuple
import pdb
from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd
import argparse
import sys
import os
import gc
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pointbiserialr

def load_examples(file_path):
    """Load examples from JSONL file"""
    print(f"Loading examples from {file_path}...")
    
    try:
        examples = []
        with open(file_path, 'r') as f:
            for line in f:
                data = json.loads(line)
                examples.append(data)
        
        print(f"Loaded {len(examples)} examples")
        return examples
    except Exception as e:
        print(f"Error loading examples: {e}")
        sys.exit(1)

def setup_models(model_name, hf_

In [6]:
# Check available models in HF_HOME
hf_hub_path = '/net/projects2/chai-lab/shared_models/hub'
import os

# List available models
models = os.listdir(hf_hub_path)
print("Available models in HF cache:")
for m in sorted(models):
    if 'models--' in m:
        print(f"  {m}")

Available models in HF cache:
  models--BAAI--bge-base-en-v1.5
  models--EleutherAI--gpt-j-6B
  models--EleutherAI--gpt-j-6b
  models--EleutherAI--gpt-neo-1.3B
  models--EleutherAI--gpt-neo-125M
  models--EleutherAI--pythia-1.4b
  models--EleutherAI--pythia-2.8b
  models--EleutherAI--pythia-410m
  models--EleutherAI--pythia-6.9b
  models--HuggingFaceH4--zephyr-7b-beta
  models--NeelNanda--Attn_Only_2L512W_C4_Code
  models--NeelNanda--Attn_Only_4L512W_C4_Code
  models--NeelNanda--gpt-neox-tokenizer-digits
  models--Qwen--Qwen2-0.5B
  models--Qwen--Qwen2.5-0.5B
  models--Qwen--Qwen2.5-0.5B-Instruct
  models--Qwen--Qwen2.5-1.5B
  models--Qwen--Qwen2.5-32B-Instruct
  models--Qwen--Qwen2.5-7B-Instruct
  models--Qwen--Qwen3-0.6B
  models--Qwen--Qwen3-32B
  models--baulab--elm-zephyr-7b-beta
  models--bert-base-uncased
  models--google--gemma-2-27b-it
  models--google--gemma-2-2b
  models--google--gemma-2-2b-it
  models--google--gemma-2-9b-it
  models--google--gemma-2b
  models--gpt2
  models

In [7]:
# Load test data to understand the format
import json

test_path = os.path.join(repo_path, 'datasets/test/test_w_chunk_score_qwen06b.json')
with open(test_path, 'r') as f:
    test_data = json.load(f)

print(f"Number of test examples: {len(test_data)}")
print(f"\nExample keys: {list(test_data[0].keys())}")
print(f"\nFirst example structure:")
for key in list(test_data[0].keys())[:8]:
    val = test_data[0][key]
    if isinstance(val, str) and len(val) > 200:
        print(f"  {key}: {val[:200]}...")
    elif isinstance(val, list) and len(val) > 3:
        print(f"  {key}: {val[:3]}... (total {len(val)} items)")
    else:
        print(f"  {key}: {val}")

Number of test examples: 256

Example keys: ['id', 'question', 'documents', 'documents_sentences', 'prompt', 'prompt_spans', 'num_tokens', 'response', 'response_spans', 'labels', 'hallucinated_llama-4-maverick-17b-128e-instruct', 'hallucinated_gpt-oss-120b', 'labels_llama', 'labels_gpt', 'scores']

First example structure:
  id: finqa_6345
  question: what is the rate of return in cadence design systems inc . of an investment from 2010 to 2011?
  documents: ['stockholder return performance graph the following graph compares the cumulative 5-year total stockholder return on our common stock relative to the cumulative total return of the nasdaq composite index and the s&p 400 information technology index . the graph assumes that the value of the investment in our common stock on january 2 , 2010 and in each index on december 31 , 2009 ( including reinvestment of dividends ) was $ 100 and tracks it each year thereafter on the last day of cadence 2019s fiscal year through january 3 , 2015 

In [8]:
# Check the scores structure
print("Scores structure for first example:")
scores = test_data[0]['scores']
print(f"Number of spans: {len(scores)}")
print(f"\nFirst span score structure:")
first_score = scores[0]
print(f"Keys: {list(first_score.keys())}")
print(f"hallucination_label: {first_score['hallucination_label']}")
print(f"r_span: {first_score['r_span']}")
print(f"\nPrompt attention scores (ECS) - sample keys:")
ecs_keys = list(first_score['prompt_attention_score'].keys())[:5]
for k in ecs_keys:
    print(f"  {k}: {first_score['prompt_attention_score'][k]}")
print(f"\nParameter knowledge scores (PKS) - sample keys:")
pks_keys = list(first_score['parameter_knowledge_scores'].keys())[:5]
for k in pks_keys:
    print(f"  {k}: {first_score['parameter_knowledge_scores'][k]}")

Scores structure for first example:
Number of spans: 5

First span score structure:
Keys: ['prompt_attention_score', 'r_span', 'hallucination_label', 'parameter_knowledge_scores']
hallucination_label: 0
r_span: [658, 701]

Prompt attention scores (ECS) - sample keys:
  (0, 0): 0.641965389251709
  (0, 1): 0.641965389251709
  (0, 2): 0.641965389251709
  (0, 3): 0.8941237926483154
  (0, 4): 0.641965389251709

Parameter knowledge scores (PKS) - sample keys:
  layer_0: 1.1865386962890625
  layer_1: 1.3424911499023438
  layer_2: 1.5926895141601562
  layer_3: 3.22711181640625
  layer_4: 4.718292236328125


## Understanding the Research

### Key Findings from the Repository:
1. **External Context Score (ECS)**: Measures how much the model attends to external context via attention heads. Computed as cosine similarity between response and context embeddings for the most attended context chunk.
2. **Parametric Knowledge Score (PKS)**: Measures Jensen-Shannon divergence between vocabulary distributions before and after FFN layers. Higher PKS indicates more parametric knowledge injection.
3. **Model Used**: Qwen3-0.6B for extracting mechanistic signals
4. **Task**: RAG hallucination detection on FinQA dataset

### Neuron-Level Finding:
- Later-layer FFNs show higher PKS for hallucinated responses (positive correlation)
- All attention heads show negative correlation between ECS and hallucination (hallucinated responses utilize less external context)

---

## GT1: Model Generalization Test

Testing if the ECS/PKS correlation findings generalize to a new model not used in the original work.

**Original Model**: Qwen3-0.6B (28 layers, 16 attention heads)
**New Model for Testing**: Pythia-1.4B (24 layers, 16 attention heads)

In [9]:
# Load required libraries for GT1 evaluation
import torch
from transformers import AutoTokenizer
from transformer_lens import HookedTransformer
from sentence_transformers import SentenceTransformer
import numpy as np
from torch.nn import functional as F
import json

print("CUDA available:", torch.cuda.is_available())
print("Loading models for GT1 evaluation...")

/home/smallyan/.conda/envs/meta/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


CUDA available: True
Loading models for GT1 evaluation...


In [10]:
# Load Pythia-1.4B model for GT1 evaluation
# This model was NOT used in the original research (which used Qwen3-0.6B)

pythia_model = HookedTransformer.from_pretrained(
    "EleutherAI/pythia-1.4b",
    device="cuda",
    torch_dtype=torch.float16
)
pythia_tokenizer = AutoTokenizer.from_pretrained("EleutherAI/pythia-1.4b")

print(f"Pythia-1.4B loaded successfully")
print(f"Number of layers: {pythia_model.cfg.n_layers}")
print(f"Number of attention heads: {pythia_model.cfg.n_heads}")
print(f"Context length: {pythia_model.cfg.n_ctx}")

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model EleutherAI/pythia-1.4b into HookedTransformer


Pythia-1.4B loaded successfully
Number of layers: 24
Number of attention heads: 16
Context length: 2048


In [11]:
# Load sentence transformer for ECS computation
bge_model = SentenceTransformer("BAAI/bge-base-en-v1.5").to("cuda")
print("BGE model loaded successfully")

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

In [12]:
# Define helper functions for GT1 evaluation

def calculate_dist_2d(sep_vocabulary_dist, sep_attention_dist):
    """Calculate Jensen-Shannon divergence between distributions (PKS)"""
    softmax_mature_layer = F.softmax(sep_vocabulary_dist, dim=-1)
    softmax_anchor_layer = F.softmax(sep_attention_dist, dim=-1)
    M = 0.5 * (softmax_mature_layer + softmax_anchor_layer)
    log_softmax_mature_layer = F.log_softmax(sep_vocabulary_dist, dim=-1)
    log_softmax_anchor_layer = F.log_softmax(sep_attention_dist, dim=-1)
    kl1 = F.kl_div(log_softmax_mature_layer, M, reduction='none').sum(dim=-1)
    kl2 = F.kl_div(log_softmax_anchor_layer, M, reduction='none').sum(dim=-1)
    js_divs = 0.5 * (kl1 + kl2)
    scores = js_divs.cpu().tolist()
    return sum(scores)

def calculate_sentence_similarity(bge_model, r_text, p_text):
    """Calculate sentence similarity using BGE model (ECS)"""
    part_embedding = bge_model.encode([r_text], normalize_embeddings=True)
    q_embeddings = bge_model.encode([p_text], normalize_embeddings=True)
    scores_named = np.matmul(q_embeddings, part_embedding.T).flatten()
    return float(scores_named[0])

print("Helper functions defined")

In [13]:
# Select test examples with known hallucination labels from the original dataset
# We'll use the test data that already has ground truth labels

# Find examples with hallucinations and without hallucinations
hallucinated_examples = []
truthful_examples = []

for ex in test_data:
    has_hallucination = any(s['hallucination_label'] == 1 for s in ex['scores'])
    if has_hallucination:
        hallucinated_examples.append(ex)
    else:
        truthful_examples.append(ex)

print(f"Examples with hallucinations: {len(hallucinated_examples)}")
print(f"Examples without hallucinations: {len(truthful_examples)}")

In [14]:
print(f"Examples with hallucinations: {len(hallucinated_examples)}")
print(f"Examples without hallucinations: {len(truthful_examples)}")

In [15]:
# Force print
import sys
sys.stdout.flush()

print(f"Examples with hallucinations: {len(hallucinated_examples)}", flush=True)
print(f"Examples without hallucinations: {len(truthful_examples)}", flush=True)

In [16]:
len(hallucinated_examples), len(truthful_examples)

In [17]:
# Check if BGE model loaded
print("BGE model loaded successfully")
print(f"Test data has {len(test_data)} examples")

In [18]:
# Something seems off with output - let's check
result = len(hallucinated_examples)
result2 = len(truthful_examples)
f"Hallucinated: {result}, Truthful: {result2}"

In [19]:
# Try direct return
len(test_data)

In [20]:
display(f"Test data: {len(test_data)}")

In [21]:
from IPython.display import display
display(f"Test data: {len(test_data)}")

In [22]:
import IPython
IPython.display.display(IPython.display.Pretty(f"Test data: {len(test_data)}"))

In [23]:
# Let me retry with more explicit output
import warnings
warnings.filterwarnings('ignore')

# Check test data again
test_count = len(test_data)
hallucinated_count = 0
truthful_count = 0

for ex in test_data:
    has_hallucination = any(s['hallucination_label'] == 1 for s in ex['scores'])
    if has_hallucination:
        hallucinated_count += 1
    else:
        truthful_count += 1

msg = f"""
Dataset Statistics:
- Total test examples: {test_count}
- Examples with hallucinations: {hallucinated_count}
- Examples without hallucinations: {truthful_count}
"""
msg

In [24]:
1+1